In [1]:
pip install llama-index-llms-huggingface

  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
    --------------------------------------- 0.3/11.9 MB ? eta -:--:--
    --------------------------------------- 0.3/11.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.9 MB 882.6 kB/s eta 0:00:13
   -- ------------------------------------- 0.8/11.9 MB 932.9 kB/s eta 0:00:12
   --- ------------------------------------ 1.0/11.9 MB 986.7 kB/s eta 0:00:12
   ---- ----------------------------------- 1.3/11.9 MB 1.0 MB/s eta 0:00:11
   ----- ---------------------------------- 1.6/11.9 MB 1.1 MB/s eta 0:00:10
   ------ --------------------------------- 1.8/11.9 MB 1.1 MB/s eta 0:00:10
   ------- -------------------------------- 2.1/11.9 MB 1.1 MB/s eta 0:00:09
   ------- ---------------------

In [3]:
pip install llama-index


   ---------------------------------------- 3/3 [llama-index]

Note: you may need to restart the kernel to use updated packages.


In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.huggingface import HuggingFaceLLM

c:\Users\LOQ\anaconda3\envs\Rag_Env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
!mkdir data

In [2]:
# load documents
documents = SimpleDirectoryReader("./data/").load_data()

In [ ]:
print(documents)

[Document(id_='a1b0a33f-c6bb-4660-b2f1-e55601d10d94', embedding=None, metadata={'file_path': 'c:\\Users\\LOQ\\Desktop\\Projects\\RAG-Sysytems\\data\\Book.pdf', 'file_name': 'Book.pdf', 'file_type': 'application/pdf', 'file_size': 6092974, 'creation_date': '2026-07-25', 'last_modified_date': '2026-07-16'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='%PDF-1.7\r\n%\r\n1 0 obj\r\n<</Type/Catalog/Pages 2 0 R/Lang(en) /StructTreeRoot 1004 0 R/MarkInfo<</Marked true>>/Metadata 7751 0 R/ViewerPreferences 7752 0 R>>\r\nendobj\r\n2 0 obj\r\n<</Type/Pages/Count 300/Kids[ 3 0 R 16 0 R 18 0 R 20 0 R 22 0 R 24 0 R 31 0 R 33 0 R 35 0 R 37 0 R 44 0 R 

In [3]:
# setup prompts - specific to StableLM
from llama_index.core import PromptTemplate

system_prompt = """<|SYSTEM|># You are a Q&A assistant. Your goal is to answer questions as
accurately as possible based on the instructions and context provided.
"""

# This will wrap the default prompts that are internal to llama-index
query_wrapper_prompt = PromptTemplate("<|USER|>{query_str}<|ASSISTANT|>")

In [ ]:
import torch

llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.7, "do_sample": False}, #but temperature only matters when do_sample=True.
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name="mistralai/Mistral-7B-Instruct-v0.1",
    model_name="mistralai/Mistral-7B-Instruct-v0.1",
    device_map="auto",
    stopping_ids=[50278, 50279, 50277, 1, 0],
    tokenizer_kwargs={"max_length": 4096},
    # uncomment this if using CUDA to reduce memory usage
    model_kwargs={"torch_dtype": torch.float16}
)

c:\Users\LOQ\anaconda3\envs\Rag_Env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LOQ\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?

In [ ]:
%pip install llama-index-embeddings-huggingface
%pip install llama-index-embeddings-instructor

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model =HuggingFaceEmbedding(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
from llama_index.core import VectorStoreIndex, ServiceContext

service_context = ServiceContext.from_defaults(
    chunk_size=1024,
    llm=llm,
    embed_model=embed_model
)

In [ ]:
index = VectorStoreIndex.from_documents(documents, service_context=service_context)

In [ ]:
query_engine = index.as_query_engine()